# Κράτηση Ξενοδοχείου με Middleware Μέλους Προτεραιότητας

Αυτό το σημειωματάριο παρουσιάζει **middleware βασισμένο σε συναρτήσεις** χρησιμοποιώντας το Microsoft Agent Framework. Βασιζόμαστε στο παράδειγμα ροής εργασίας με συνθήκες προσθέτοντας ένα επίπεδο middleware που προσφέρει ειδικά προνόμια στα μέλη προτεραιότητας.

## Τι θα μάθετε:
1. **Middleware Βασισμένο σε Συναρτήσεις**: Παρεμβολή και τροποποίηση αποτελεσμάτων συναρτήσεων  
2. **Πρόσβαση σε Context**: Ανάγνωση και τροποποίηση του `context.result` μετά την εκτέλεση  
3. **Υλοποίηση Επιχειρηματικής Λογικής**: Οφέλη μελών προτεραιότητας  
4. **Αλλαγή Αποτελεσμάτων**: Επικύρωση αποτελεσμάτων συναρτήσεων βάσει κατάστασης χρήστη  
5. **Ίδια Ροή Εργασίας, Διαφορετικά Αποτελέσματα**: Αλλαγές συμπεριφοράς καθοριζόμενες από middleware  

## Αρχιτεκτονική Ροής Εργασίας με Middleware:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## Κύρια Διαφορά από τη Ροή Εργασίας με Συνθήκες:

**Χωρίς Middleware** (14-conditional-workflow.ipynb):
- Παρίσι δεν έχει δωμάτια → Δρομολόγηση στο alternative_agent

**Με Middleware** (αυτό το σημειωματάριο):
- Κανονικός χρήστης + Παρίσι → Δεν υπάρχουν δωμάτια → Δρομολόγηση στο alternative_agent  
- Χρήστης προτεραιότητας + Παρίσι → 🌟 Το Middleware κάνει υπέρβαση! → Διαθέσιμο → Δρομολόγηση στο booking_agent  

## Απαιτούμενα:
- Εγκατεστημένο Microsoft Agent Framework  
- Κατανόηση ροών εργασίας με συνθήκες (δείτε το 14-conditional-workflow.ipynb)  
- Διακριτικό GitHub ή κλειδί API OpenAI  
- Βασική κατανόηση προτύπων middleware


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Βήμα 1: Ορισμός Μοντέλων Pydantic για Δομημένα Αποτελέσματα

Αυτά τα μοντέλα ορίζουν το **σχήμα** που θα επιστρέφουν οι πράκτορες. Έχουμε προσθέσει ένα πεδίο `priority_override` για να παρακολουθούμε πότε το middleware τροποποιεί το αποτέλεσμα διαθεσιμότητας.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Step 2: Ορισμός Βάσης Δεδομένων Μελών Προτεραιότητας

Για αυτή τη δοκιμή, θα προσομοιώσουμε μια βάση δεδομένων μελών προτεραιότητας. Σε παραγωγή, αυτό θα έκανε ερώτημα σε μια πραγματική βάση δεδομένων ή API.

**Μέλη Προτεραιότητας:**
- `alice@example.com` - Μέλος VIP
- `bob@example.com` - Premium μέλος  
- `priority_user` - Λογαριασμός δοκιμής


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## Βήμα 3: Δημιουργήστε το Εργαλείο Κράτησης Ξενοδοχείου

Ίδιο με τη ροή εργασίας υπό όρους, αλλά τώρα θα αναχαιτιστεί από ενδιάμεσο λογισμικό!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Step 4: 🌟 Δημιουργία Middleware Ελέγχου Προτεραιότητας (ΚΛΕΙΔΙΚΟ ΧΑΡΑΚΤΗΡΙΣΤΙΚΟ!)

Αυτή είναι η **βασική λειτουργικότητα** αυτού του σημειωματάριου. Το middleware:

1. **Αποκλείει** την κλήση της συνάρτησης hotel_booking
2. **Εκτελεί** τη συνάρτηση κανονικά καλώντας `next(context)`
3. **Ελέγχει** το αποτέλεσμα στο `context.result`
4. **Αντικαθιστά** το αποτέλεσμα αν ο χρήστης έχει προτεραιότητα και δεν υπάρχουν διαθέσιμα δωμάτια
5. **Επιστρέφει** το τροποποιημένο αποτέλεσμα πίσω στον πράκτορα

**Κύριο Πρότυπο:**  
```python
async def my_middleware(context, next):
    await next(context)  # Εκτέλεση συνάρτησης
    # Τώρα το context.result περιέχει το αποτέλεσμα της συνάρτησης
    if some_condition:
        context.result = new_value  # Παράκαμψη!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## Βήμα 5: Ορισμός Συναρτήσεων Συνθηκών για τον Καθορισμό Διαδρομής

Οι ίδιες συναρτήσεις συνθηκών όπως στη ροή εργασίας υπό συνθήκη - ελέγχουν την δομημένη έξοδο για να καθορίσουν τη διαδρομή.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## Step 6: Δημιουργία Προσαρμοσμένου Εκτελεστή Εμφάνισης

Ίδιος εκτελεστής όπως πριν - εμφανίζει το τελικό αποτέλεσμα της ροής εργασίας.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## Step 7: Φόρτωση Μεταβλητών Περιβάλλοντος

Ρυθμίστε τον πελάτη LLM (GitHub Models ή OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Βήμα 8: Δημιουργία Πρακτόρων AI με Middleware

**ΚΥΡΙΑ ΔΙΑΦΟΡΑ:** Όταν δημιουργούμε τον availability_agent, περνάμε την παράμετρο `middleware`!

Έτσι γίνεται η έγχυση του priority_check_middleware στην αλυσίδα κλήσεων της λειτουργίας του πράκτορα.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Step 9: Δημιουργία της Ροής Εργασίας

Ίδια δομή ροής εργασίας όπως πριν - προσανατολισμός υπό όρους με βάση τη διαθεσιμότητα.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## Βήμα 10: Περίπτωση δοκιμής 1 - Κανονικός χρήστης στο Παρίσι (Χωρίς Υπέρβαση)

Ένας κανονικός χρήστης προσπαθεί να κλείσει Παρίσι → Καμία διαθεσιμότητα δωματίων → Κατευθύνεται στον alternative_agent


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## Step 11: Test Case 2 - 🌟 Προτεραιότητα Χρήστη στο Παρίσι (ΜΕ παράκαμψη!)

Ένα μέλος προτεραιότητας προσπαθεί να κάνει κράτηση Παρίσι → Αρχικά δεν υπάρχουν διαθέσιμα δωμάτια → 🌟 Το middleware κάνει παράκαμψη! → Δρομολογεί στον booking_agent

**Αυτή είναι η βασική επίδειξη της δύναμης του middleware!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## Βήμα 12: Περίπτωση Δοκιμής 3 - Χρήστης με Προτεραιότητα στη Στοκχόλμη (Ήδη Διαθέσιμο)

Ο χρήστης με προτεραιότητα δοκιμάζει Στοκχόλμη → Διαθέσιμα δωμάτια → Δεν απαιτείται αντικατάσταση → Κατευθύνεται στο booking_agent

Αυτό δείχνει ότι το middleware ενεργεί μόνο όταν χρειάζεται!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## Βασικά Συμπεράσματα και Έννοιες Middleware

### ✅ Τι Μάθατε:

#### **1. Πρότυπο Middleware με Βάση τις Συναρτήσεις**

Το middleware παρεμβάλλεται στις κλήσεις συναρτήσεων χρησιμοποιώντας μια απλή ασύγχρονη συνάρτηση:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # Πριν από την εκτέλεση της συνάρτησης
    print("Intercepting...")
    
    # Εκτελέστε τη συνάρτηση
    await next(context)
    
    # Μετά την εκτέλεση της συνάρτησης - ελέγξτε το αποτέλεσμα
    if context.result:
        # Τροποποιήστε το αποτέλεσμα αν χρειάζεται
        context.result = modified_value
```

#### **2. Πρόσβαση στο Context και Επικαλύψεις Αποτελεσμάτων**

- `context.function` - Πρόσβαση στη συνάρτηση που καλείται
- `context.arguments` - Ανάγνωση των ορισμάτων της συνάρτησης
- `context.kwargs` - Πρόσβαση σε επιπλέον παραμέτρους
- `await next(context)` - Εκτέλεση της συνάρτησης
- `context.result` - Ανάγνωση/τροποποίηση της εξόδου της συνάρτησης

#### **3. Υλοποίηση Επιχειρηματικής Λογικής**

Το middleware μας υλοποιεί προνόμια μελών με προτεραιότητα:
- **Κανονικοί χρήστες**: Χωρίς τροποποιήσεις, τυπική ροή εργασιών
- **Χρήστες προτεραιότητας**: Επικαλύπτει το "καμία διαθεσιμότητα" → "διαθέσιμο"
- **Υπό όρους λογική**: Επικαλύψεις μόνο όταν απαιτείται

#### **4. Ίδια Ροή Εργασίας, Διαφορετικά Αποτελέσματα**

Η δύναμη του middleware:
- ✅ Καμία αλλαγή στη δομή της ροής εργασίας
- ✅ Καμία αλλαγή στη λειτουργία του εργαλείου
- ✅ Καμία αλλαγή στη λογική δρομολόγησης υπό συνθήκες
- ✅ Μόνο middleware → Διαφορετική συμπεριφορά!

### 🚀 Εφαρμογές στον Πραγματικό Κόσμο:

1. **Λειτουργίες VIP/Premium**
   - Επικαλύπτει τους περιορισμούς ρυθμών για premium χρήστες
   - Παρέχει προτεραιότητα πρόσβασης σε πόρους
   - Αποκλεισμός premium λειτουργιών δυναμικά

2. **A/B Testing**
   - Δρομολόγηση χρηστών σε διαφορετικές υλοποιήσεις
   - Δοκιμή νέων λειτουργιών με συγκεκριμένους χρήστες
   - Σταδιακές κυκλοφορίες λειτουργιών

3. **Ασφάλεια & Συμμόρφωση**
   - Έλεγχος κλήσεων συναρτήσεων
   - Αποκλεισμός ευαίσθητων λειτουργιών
   - Εφαρμογή επιχειρηματικών κανόνων

4. **Βελτιστοποίηση Απόδοσης**
   - Αποθήκευση αποτελεσμάτων για συγκεκριμένους χρήστες
   - Παράλειψη δαπανηρών λειτουργιών όπου είναι δυνατό
   - Δυναμική κατανομή πόρων

5. **Διαχείριση Σφαλμάτων & Επαναδοκιμή**
   - Προσέγγιση και ευγενής διαχείριση σφαλμάτων
   - Υλοποίηση λογικής επαναδοκιμής
   - Εναλλακτικές λύσεις ως πλάγια επιλογή

6. **Καταγραφή & Παρακολούθηση**
   - Παρακολούθηση χρόνων εκτέλεσης συναρτήσεων
   - Καταγραφή παραμέτρων και αποτελεσμάτων
   - Παρακολούθηση προτύπων χρήσης

### 🔑 Κύριες Διαφορές από τα Decorators:

| Χαρακτηριστικό | Decorator | Middleware |
|----------------|-----------|------------|
| **Εύρος** | Μια συνάρτηση | Όλες οι συναρτήσεις στον agent |
| **Ευελιξία** | Σταθερό κατά τον ορισμό | Δυναμικό κατά την εκτέλεση |
| **Context** | Περιορισμένο | Πλήρες context του agent |
| **Σύνθεση** | Πολλαπλά decorators | Σωλήνας middleware |
| **Ενημέρωση Agent** | Όχι | Ναι (πρόσβαση στην κατάσταση agent) |

### 📚 Πότε να Χρησιμοποιήσετε Middleware:

✅ **Χρησιμοποιήστε middleware όταν:**
- Χρειάζεται να τροποποιήσετε τη συμπεριφορά ανάλογα με κατάσταση χρήστη/συνεδρίας
- Θέλετε να εφαρμόσετε λογική σε πολλές συναρτήσεις
- Χρειάζεστε πρόσβαση σε context επιπέδου agent
- Υλοποιείτε διατομεακά ζητήματα (καταγραφή, έλεγχος πρόσβασης κλπ.)

❌ **Μην χρησιμοποιείτε middleware όταν:**
- Απλός έλεγχος εισόδου (χρησιμοποιείστε Pydantic)
- Λογική ειδική για συνάρτηση (κρατήστε τη στη συνάρτηση)
- Μόνο μια φορά τροποποιήσεις (αλλάξτε απλώς τη συνάρτηση)

### 🎓 Προχωρημένα Πρότυπα:

```python
# Πολλαπλά middleware (η σειρά εκτέλεσης έχει σημασία!)
middleware=[
    logging_middleware,      # Πρώτα καταγραφές
    auth_middleware,         # Έπειτα έλεγχος ταυτότητας
    cache_middleware,        # Έπειτα έλεγχος cache
    rate_limit_middleware,   # Έπειτα περιορισμοί ρυθμού
    priority_check_middleware  # Τέλος έλεγχος προτεραιότητας
]

# Υπό όρους εκτέλεση middleware
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # Τροποποίηση αποτελέσματος
    else:
        # Παράλειψη εκτέλεσης εντελώς
        context.result = cached_value
```

### 🔗 Σχετικές Έννοιες:

- **Agent Middleware**: Παρεμβαίνει στις κλήσεις agent.run()
- **Function Middleware**: Παρεμβαίνει στις κλήσεις συναρτήσεων εργαλείων (αυτή που χρησιμοποιήσαμε!)
- **Middleware Pipeline**: Αλυσίδα middleware που εκτελείται με σειρά
- **Διάδοση Context**: Μεταφορά κατάστασης μέσω της αλυσίδας middleware


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
